# Teste do setup do TFlite

Este notebook documenta um passo a passo simples para preparar e testar um modelo TFLite (MobileNetV2 quantizado) em um ambiente como Raspberry Pi. O objetivo é baixar o modelo e os rótulos, verificar dependências e criar um interpretador TFLite pronto para inferência.

O que este notebook faz:
- Verifica/importa as bibliotecas necessárias (tflite_runtime, NumPy, Pillow).
- Checa versões das bibliotecas instaladas.
- Baixa e extrai o modelo quantizado MobileNetV2 (.tflite) em ./models.
- Baixa o arquivo labels.txt com os rótulos das classes.
- Lista e organiza os arquivos no diretório de modelos (apenas .tflite e labels.txt são necessários).
- Define o caminho do modelo e cria o interpretador TFLite, alocando tensores.

Requisitos:
- - Se você ainda não instalou o TensorFlow Lite, siga as instruções na seção "Instalando o TensorFlow Lite no Raspberry Pi" do roteiro de laboratório [Instalação de Bibliotecas Python para o RPi](https://github.com/fabiobento/sis-emb-2025-2/blob/main/aulas/sbc-rpi/rpi_ei_linux_sdk/rpi_ei_linux_sdk.md)

Próximos passos típicos (não todos implementados aqui):
- Pré-processar imagens para o tamanho/normalização esperados (224x224).
- Carregar a imagem como array NumPy e ajustar para o tipo de entrada do modelo (uint8 para modelo quantizado).
- Executar inferência com interpreter.set_tensor(...); interpreter.invoke(); e ler a saída com interpreter.get_tensor(...).
- Mapear as probabilidades para labels usando labels.txt e exibir as top-k previsões.

Observações práticas:
- Trabalhe no diretório do notebook para que ./models funcione corretamente.
- Em dispositivos com espaço limitado, mantenha apenas o .tflite e labels.txt.
- Se ocorrerem erros de permissão ao baixar/extrair, verifique permissões do diretório e conectividade de rede.

## Importar bibliotecas

In [ ]:
import tflite_runtime.interpreter as tflite
import numpy as np
from PIL import Image

## Verificar as versões das bibliotecas instaladas

In [ ]:
print("NumPy:", np.__version__)
print("Pillow:", Image.__version__)

## Baixar modelo pré-treinado


- Baixe o modelo pré-treinado MobileNetV2    

    - Um modelo pré-treinado adequado é muito importante para o sucesso da classificação de imagens em dispositivos com recursos limitados, como o Raspberry Pi.
    - O [*MobileNet*](https://github.com/tensorflow/models/tree/master/research/slim/nets/mobilenet) foi projetado para aplicações móveis e de visão embarcada, com um bom equilíbrio entre precisão e velocidade
    
    - Várias versões estão disponíveis: `MobileNetV1`, `MobileNetV2`, `MobileNetV3`.
- Vamos baixar a V2:

In [ ]:
from pathlib import Path
import tarfile
import urllib.request

# Define o diretório onde os modelos serão salvos
models_dir = Path("./models")
models_dir.mkdir(parents=True, exist_ok=True)

# Define os caminhos dos arquivos do modelo e do arquivo compactado
tflite_file = models_dir / "mobilenet_v2_1.0_224_quant.tflite"
tgz_file = models_dir / "mobilenet_v2_1.0_224_quant.tgz"
url = "https://storage.googleapis.com/download.tensorflow.org/models/tflite_11_05_08/mobilenet_v2_1.0_224_quant.tgz"

# Verifica se o arquivo .tflite já existe
if tflite_file.exists():
    print(f"{tflite_file} já existe. Pulando download e extração.")
else:
    # Se o arquivo .tgz não existe, faz o download
    if not tgz_file.exists():
        print(f"Baixando {url} para {tgz_file} ...")
        urllib.request.urlretrieve(url, tgz_file)
    else:
        print(f"{tgz_file} já existe. Pulando download.")
    try:
        # Extrai o arquivo .tgz para o diretório de modelos
        print(f"Extraindo {tgz_file} para {models_dir} ...")
        with tarfile.open(tgz_file, "r:gz") as tar:
            tar.extractall(path=models_dir)
        print("Extração concluída.")
    except Exception as e:
        print("Falha ao extrair:", e)

# Verifica novamente se o arquivo .tflite está disponível
if tflite_file.exists():
    print("Arquivo .tflite pronto:", tflite_file)
else:
    print("Arquivo .tflite não encontrado após extração.")

- Agora faça upload do rótulos (labels) das classes para o seu RPi

In [ ]:
# Instala o pacote gdown para baixar arquivos do Google Drive
!pip install gdown

# Baixa o arquivo labels.txt do Google Drive para a pasta models
!gdown --id 1TYjT6OMGzBKkZuO-oq6Jac8eO7QtH5qC -O ./models/labels.txt

print("Download concluído!")

- Liste os arquivos do diretório `models`:

In [ ]:
ls ~/Documents/TFLITE/IMG_CLASS/models

Você verá algo parecido com:

```bash
2_teste_setup.ipynb
mobilenet_v2_1.0_224_quant.ckpt.data-00000-of-00001
mobilenet_v2_1.0_224_quant.ckpt.index
mobilenet_v2_1.0_224_quant.ckpt.meta
mobilenet_v2_1.0_224_quant.tflite
mobilenet_v2_1.0_224_quant.tgz
mobilenet_v2_1.0_224_quant_eval.pbtxt
mobilenet_v2_1.0_224_quant_frozen.pb
mobilenet_v2_1.0_224_quant_info.txt
```

- No entanto, apenas precisamos apenas do modelo `mobilenet_v2_1.0_224_quant.tflite` e do arquivo `labels.txt` com os rótulos das classes.
    - Você pode apagar os outros arquivos baixados.
- O arquivo `labels.txt` contém os rótulos das **1001** classes do modelo `MobileNetV2`, que são usados para interpretar as previsões do modelo.

In [ ]:
model_path = "./models/mobilenet_v2_1.0_224_quant.tflite"

## Criar interpretador TFLite

- O interpretador TFLite é necessário para carregar e executar o modelo
    - Por isso testamos abaixo o método `allocate_tensors()` que aloca  memória para os tensores de entrada e saída

In [ ]:
# Experimente criar um interpretador TFLite
interpreter = tflite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()
print("Interpretador TFLite criado com sucesso!")